In [10]:
import os

from PIL import Image
from PIL import ImageDraw
# Import necessary libraries
from transformers import pipeline

# Build the object-detection pipeline using 🤗 Transformers Library
od_pipe = pipeline(task="object-detection", model="facebook/detr-resnet-50")


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [18]:
# Define helper functions
def load_image_from_path(image_path):
    img = Image.open(image_path)
    return img


def render_results_in_image(image, results):
    draw = ImageDraw.Draw(image)
    for result in results:
        box = result['box']
        label = result['label']
        score = result['score']
        draw.rectangle([(box['xmin'], box['ymin']), (box['xmax'], box['ymax'])], outline="red", width=3)
        draw.text((box['xmin'], box['ymin']), f"{label} ({score:.2f})", fill="red")
    return image


# Load Image
image_path = "img_test.jpg"
raw_image = load_image_from_path(image_path)

# Resize image
raw_image = raw_image.resize((600, 400))

# Detect objects in the image
pipeline_output = od_pipe(raw_image)

# Render results on the image
propossed_image = render_results_in_image(raw_image.copy(), pipeline_output)


# Save cropped objects
def crop_and_save_objects(image, pipeline_output, save_dir="cropped_objects"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for i, result in enumerate(pipeline_output):
        box = result['box']
        label = result['label']
        cropped_image = image.crop((box['xmin'], box['ymin'], box['xmax'], box['ymax']))
        path_save = os.path.join(save_dir, f"{label}_{i}.png")
        cropped_image.save(path_save)


crop_and_save_objects(raw_image, pipeline_output)